In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import re

In [2]:
# URL to scrape
url = 'https://en.wikipedia.org/wiki/Timeline_of_the_20th_century'

# Send a GET request to fetch the webpage content
response = requests.get(url)

# Parse the content using BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Output file path
output_file = 'old_timeline.csv'

# Define a regex pattern to clean event descriptions
date_pattern = re.compile(
    r'^[•-]?\s?[A-Za-z]+(\s\d{1,2})+(\s?-\s?\d{1,2})?:?\s?:?|'
    r'[•-]?\s?–\s?\d{1,2}+:?\s?|'
    r'^[•-]?\s?[A-Za-z]+(\s-\s[A-Za-z]+)?:?\s?:?|'
    r'[•-]?\s?–\s[A-Za-z]+:?\s?|' 
    r'^[•-]?\s?[A-Za-z]+\s\d{1,2}:?\s?'
)  # Pattern to match "Month", "Month day1 - day2", "Month1 - month2", "– Month", "Month day"

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Date', 'Event Description'])  # Write the header row

    current_year = None  # Track the current year
    start_processing = False  # Flag to start processing relevant sections

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'p', 'ul']):
        # Stop parsing if we reach sections like "See also" or "References"
        if 'Horizontal timelines' in section.text or 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text:
            break

        # Extract potential decade or specific year from headings (like <h2>, <h3>, etc.)
        if section.name in ['h2', 'h3']:
            heading_text = section.text.strip()

            # Skip irrelevant century and decade headings
            if 'century' in heading_text.lower() or 'decade' in heading_text.lower():
                continue

            # If it's a valid year, assign it as current_year and start processing
            match = re.search(r'(\d{4})', heading_text)
            if match:
                current_year = match.group(1)
                start_processing = True
                print(f"Processing year: {current_year}")

        # Only process sections after encountering the first valid year
        if start_processing and section.name == 'ul':  # Events are often listed in bullet points
            for li in section.find_all('li'):
                event_description = li.text.strip()

                # Check if the event description starts with a date pattern
                if date_pattern.match(event_description):
                    # Clean the date part (e.g., "Month", "Month day1 - day2", "Month1 - month2", "– Month", "Month day")
                    event_description_cleaned = re.sub(date_pattern, '', event_description)
                else:
                    # Keep the entire description if no date pattern is found
                    event_description_cleaned = event_description

                # Remove leading colon and quotes if present
                event_description_cleaned = event_description_cleaned.lstrip(':').strip().strip('"')

                # Write the cleaned row to the output CSV file
                writer.writerow([current_year, event_description_cleaned])
                print(f"Writing to CSV: Date - {current_year}, Description - {event_description_cleaned}")

Processing year: 1900
Processing year: 1901
Writing to CSV: Date - 1901, Description - The Australian colonies federate.[1]
Writing to CSV: Date - 1901, Description - Edward VII became King of England and India after Queen Victoria's death.
Writing to CSV: Date - 1901, Description - The Platt Amendment provides for Cuban independence in exchange for the withdrawal of American troops.
Writing to CSV: Date - 1901, Description - Emily Hobhouse reports on the poor conditions in 45 British internment camps for Boer women and children in South Africa.
Writing to CSV: Date - 1901, Description - The assassination of William McKinley ushered in office Vice President Theodore Roosevelt after McKinley's death on September 14.
Writing to CSV: Date - 1901, Description - The Eight-Nation Alliance defeats the Boxer Rebellion, and imposes heavy financial penalties on China.
Writing to CSV: Date - 1901, Description - First Nobel Prizes awarded.
Writing to CSV: Date - 1901, Description - Guglielmo Marco

,Date,Event Description
0,1901,The Australian colonies federate.[1]
1,1901,Edward VII became King of England and India af...
2,1901,The Platt Amendment provides for Cuban indepen...
3,1901,Emily Hobhouse reports on the poor conditions ...
4,1901,The assassination of William McKinley ushered ...
5,1901,The Eight-Nation Alliance defeats the Boxer Re...
6,1901,First Nobel Prizes awarded.
7,1901,Guglielmo Marconi received the first transatla...
8,1902,The Unification of Saudi Arabia begins.
9,1902,Cuba given independence by the United States.
